In [ ]:
import znnl as nl
import jax
import jax.numpy as np
import flax.linen as nn
import numpy as onp
from flax.training import train_state
import optax
from rich.progress import track
import matplotlib.pyplot as plt
from dataclasses import dataclass
import neural_tangents as nt
from sklearn.manifold import TSNE

## Network definition

In [ ]:
def create_n(layers: int, width: int = 128):

    class Predictor(nn.Module):
        """A simple dense model."""

        @nn.compact
        def __call__(self, x):
            x = x.reshape((x.shape[0], -1))
            for _ in range(layers):
                x = nn.Dense(features=width)(x)
                x = nn.relu(x)
            x = nn.Dense(features=10)(x)
            return x
        
    return Predictor()
    

## Training functions

In [ ]:
def get_ntk_function(apply_fn, batch_size: int):
    empirical_ntk = nt.batch(
                nt.empirical_ntk_fn(
                    f=apply_fn, trace_axes=(-1,), implementation=nt.NtkImplementation.JACOBIAN_CONTRACTION
                ),
                batch_size=batch_size,
            )
    return jax.jit(empirical_ntk)

In [ ]:
def create_train_state(module, rng, learning_rate):
    """Creates an initial TrainState."""

    params = module.init(rng, np.ones([1, 28, 28, 1]))['params']

    tx = optax.adam(learning_rate)

    return train_state.TrainState.create(
      apply_fn=module.apply, params=params, tx=tx,
    )

In [ ]:
mnist_generator = nl.data.MNISTGenerator(ds_size=200)

In [ ]:
data = {}

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20]
widths = [2, 12, 50, 128, 256, 500]
ensembles = 5

for width in widths:
    entropies = {"values": [], "errors": []}
    traces = {"values": [], "errors": []}
    for depth in track(depths, description=width):
        for _ in range(ensembles):
            tmp_s = []
            tmp_tr = []

            model = create_n(depth, width=width)

            # Seed the model state.
            init_rng = jax.random.PRNGKey(onp.random.randint(76325426354))
            state = create_train_state(model, init_rng, learning_rate)
            del init_rng

            # f(x, \theta)
            def apply_fn(params, x):
                return model.apply({"params":params}, x, mutable=['batch_stats'])[0]

            ntk_fn = get_ntk_function(apply_fn, batch_size=100)

            ntk = ntk_fn(
                mnist_generator.train_ds["inputs"], 
                mnist_generator.train_ds["inputs"], 
                state.params
            )

            eigs, _ = np.linalg.eigh(ntk)
            eigs /= eigs.sum()

            tmp_s.append(-1 * (eigs * np.log(eigs)).sum())
            tmp_tr.append(np.trace(ntk))
            
        tmp_s = np.array(tmp_s)[~np.isnan(np.array(tmp_s))]
        tmp_tr = np.array(tmp_tr)[~np.isnan(np.array(tmp_tr))]
        
        entropies["values"].append(np.mean(tmp_s))
        entropies["errors"].append(
            np.std(tmp_s) / np.sqrt(len(tmp_s))
        )
        
        traces["values"].append(np.mean(tmp_tr))
        traces["errors"].append(
            np.std(tmp_tr) / np.sqrt(len(tmp_tr))
        )
        
    data[width] = {"entropy": entropies, "trace": traces}

## Analysis

In [ ]:
for item, values in data.items():
    plt.errorbar(
        [i for i in range(len(values["entropy"]["values"]))],
        values["entropy"]["values"],
        yerr=values["entropy"]["errors"],
        linestyle="none",
        marker=".",
        label=item
    )

plt.hlines(-1 * np.log(1/200), 0, 10, color="r")  # max 200 point entropy
plt.hlines(-1 * np.log(1/10), 0, 10, color="b")  # max 10 point entropy

plt.xlabel("This is not correct, it is depths from 1 to 20")

plt.legend()
plt.show()

In [ ]:
for item, values in data.items():
    plt.errorbar(
        [i for i in range(len(values["trace"]["values"]))],
        values["trace"]["values"],
        yerr=values["trace"]["errors"],
        linestyle="none",
        marker=".",
        label=item
    )

plt.xlabel("This is not correct, it is depths from 1 to 20")

plt.legend()
plt.show()